# Exam Analysis — Question difficulty and quality

This notebook analyses:
1. Answer distribution per question (stacked bars)
2. Identification of extreme questions (very easy / very difficult)
3. Discrimination index: does each question distinguish good from poor students?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#16213e',
    'axes.edgecolor': '#e94560', 'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0', 'xtick.color': '#e0e0e0',
    'ytick.color': '#e0e0e0', 'grid.color': '#2a2a4a', 'grid.alpha': 0.5,
    'font.family': 'sans-serif', 'font.size': 11,
})

COLORS = {
    'correct': '#00e676', 'wrong': '#e94560', 'unanswered': '#888888',
}

POINTS_CORRECT = 0.11
POINTS_WRONG = -0.05
PASS_THRESHOLD = 5.0

CSV_PATH = Path('.') / '134_36018173_ZE3IFC005200_MP5072_A-Examen 1a evaluación-cualificacións.csv'

def parse_answer(val) -> str:
    if isinstance(val, (int, float)):
        return 'correct' if val > 0 else ('wrong' if val < 0 else 'unanswered')
    s = str(val).strip().strip("'").strip('\u2018')
    if s in ('-', '', 'nan'):
        return 'unanswered'
    try:
        num = float(s.replace(',', '.'))
        return 'correct' if num > 0 else ('wrong' if num < 0 else 'unanswered')
    except ValueError:
        return 'unanswered'

def parse_score(val) -> float:
    if isinstance(val, (int, float)): return float(val)
    s = str(val).strip().strip("'").strip('\u2018')
    if s in ('-', '', 'nan'): return 0.0
    try: return float(s.replace(',', '.'))
    except ValueError: return 0.0

df = pd.read_csv(CSV_PATH)
df = df[df['Apelidos'].str.strip() != 'Media xeral'].dropna(subset=['Apelidos']).copy()
q_cols = [c for c in df.columns if c.startswith('P.')]
NUM_QUESTIONS = len(q_cols)
df['Alumno'] = df['Nome'].astype(str).str.strip() + ' ' + df['Apelidos'].astype(str).str.strip()
grade_col = [c for c in df.columns if 'ualificaci' in c][0]
df['Nota_Original'] = df[grade_col].apply(parse_score)

answer_types = pd.DataFrame(index=df.index)
scores = pd.DataFrame(index=df.index)
for col in q_cols:
    answer_types[col] = df[col].apply(parse_answer)
    scores[col] = df[col].apply(parse_score)

n_students = len(df)
print(f'Datos cargados: {n_students} alumnos, {NUM_QUESTIONS} preguntas')

## 1. Answer distribution per question (stacked bars)

In [ ]:
# Calcular estadísticas por pregunta
difficulties = []
for i, col in enumerate(q_cols, 1):
    n_correct = (answer_types[col] == 'correct').sum()
    n_wrong = (answer_types[col] == 'wrong').sum()
    n_unanswered = (answer_types[col] == 'unanswered').sum()
    pct_correct = n_correct / n_students * 100
    difficulties.append({
        'Q': i, 'Aciertos': n_correct, 'Errores': n_wrong,
        'Sin resp.': n_unanswered, '% Acierto': f'{pct_correct:.0f}%',
        'pct_correct': pct_correct,
    })

# Mostrar tabla
diff_df = pd.DataFrame(difficulties)
print('Resumen por pregunta:')
diff_df[['Q', 'Aciertos', 'Errores', 'Sin resp.', '% Acierto']]

In [ ]:
# Gráfica de barras apiladas
fig, ax = plt.subplots(figsize=(22, 7))
fig.suptitle('Distribución de Respuestas por Pregunta — SAA', fontweight='bold', fontsize=15)

q_nums = np.arange(1, NUM_QUESTIONS + 1)
corrects = np.array([d['Aciertos'] for d in difficulties])
wrongs = np.array([d['Errores'] for d in difficulties])
unanswered = np.array([d['Sin resp.'] for d in difficulties])

w = 0.7
ax.bar(q_nums, corrects, w, color=COLORS['correct'], label='Correctas', alpha=0.85)
ax.bar(q_nums, wrongs, w, bottom=corrects, color=COLORS['wrong'], label='Erróneas', alpha=0.85)
ax.bar(q_nums, unanswered, w, bottom=corrects + wrongs,
       color=COLORS['unanswered'], label='Sin responder', alpha=0.6)

ax.set_xlabel('Nº de Pregunta')
ax.set_ylabel('Nº de Alumnos')
ax.set_xticks(q_nums)
ax.set_xticklabels(q_nums, fontsize=6, rotation=90)
ax.set_ylim(0, n_students + 0.5)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.axhline(y=n_students * 0.25, color='white', linestyle=':', linewidth=1, alpha=0.3,
           label='25% de alumnos')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, axis='y', alpha=0.3)

# Marcar preguntas con 0% acierto
for i, d in enumerate(difficulties):
    if d['Aciertos'] == 0:
        ax.text(i + 1, n_students + 0.2, '✗', ha='center', fontsize=10,
                color=COLORS['wrong'], fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Extreme questions

In [ ]:
# Preguntas más difíciles (< 25% acierto)
difficult = diff_df[diff_df['pct_correct'] < 25].sort_values('pct_correct')
print(f"\n🔴 Preguntas con tasa de acierto < 25% ({len(difficult)}/{NUM_QUESTIONS}):")
difficult[['Q', 'Aciertos', 'Errores', 'Sin resp.', '% Acierto']]

In [ ]:
# Preguntas más fáciles (>= 75% acierto)
easy = diff_df[diff_df['pct_correct'] >= 75].sort_values('pct_correct', ascending=False)
print(f"\n🟢 Preguntas con tasa de acierto ≥ 75% ({len(easy)}/{NUM_QUESTIONS}):")
easy[['Q', 'Aciertos', 'Errores', 'Sin resp.', '% Acierto']]

## 3. Discrimination index per question

The discrimination index measures whether a question distinguishes good students from poor ones. It is calculated as the point-biserial correlation between getting the question right (1/0) and the total grade.
- **Discrimination > 0.3**: excellent question
- **0.2 – 0.3**: acceptable
- **0.0 – 0.2**: low, reviewable
- **≤ 0**: poor, should be removed or reworded

In [ ]:
# Calcular discriminación
grades = df['Nota_Original'].values
disc_data = []
for i, col in enumerate(q_cols, 1):
    correct_mask = (answer_types[col] == 'correct').astype(int).values
    if correct_mask.std() == 0:
        d = 0.0  # todos acertaron o ninguno
    else:
        d = np.corrcoef(correct_mask, grades)[0, 1]
    pct = difficulties[i-1]['pct_correct']
    quality = '✗ MALA' if d <= 0 else ('⚠ BAJA' if d < 0.2 else ('✓ OK' if d < 0.3 else '★ EXCELENTE'))
    disc_data.append({
        'Q': i, '% Acierto': f'{pct:.0f}%', 'Discriminación': round(d, 3), 'Calidad': quality
    })

disc_df = pd.DataFrame(disc_data)
disc_df

In [ ]:
# Gráfica de discriminación
fig, ax = plt.subplots(figsize=(22, 6))
fig.suptitle('Índice de Discriminación por Pregunta', fontweight='bold', fontsize=15)

disc_vals = [d['Discriminación'] for d in disc_data]
bar_colors = ['#e94560' if d <= 0 else ('#ffc947' if d < 0.2 else '#00e676') for d in disc_vals]

ax.bar(range(1, NUM_QUESTIONS + 1), disc_vals, color=bar_colors, edgecolor='white',
       linewidth=0.5, alpha=0.85)
ax.axhline(y=0.0, color='#e94560', linestyle='--', linewidth=1.5, alpha=0.7, label='Discriminación = 0')
ax.axhline(y=0.2, color='#ffc947', linestyle=':', linewidth=1.5, alpha=0.5, label='Umbral baja (0.2)')
ax.axhline(y=0.3, color='#00e676', linestyle=':', linewidth=1.5, alpha=0.5, label='Umbral buena (0.3)')
ax.set_xlabel('Nº de Pregunta')
ax.set_ylabel('Índice de Discriminación')
ax.set_xticks(range(1, NUM_QUESTIONS + 1))
ax.set_xticklabels(range(1, NUM_QUESTIONS + 1), fontsize=6, rotation=90)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

n_bad = sum(1 for d in disc_vals if d <= 0)
n_low = sum(1 for d in disc_vals if 0 < d < 0.2)
n_ok = sum(1 for d in disc_vals if 0.2 <= d < 0.3)
n_exc = sum(1 for d in disc_vals if d >= 0.3)
print(f'\nResumen de calidad: {n_bad} malas, {n_low} bajas, {n_ok} aceptables, {n_exc} excelentes')

In [ ]:
# Preguntas con discriminación negativa o nula
bad_disc = disc_df[disc_df['Discriminación'] <= 0].sort_values('Discriminación')
print(f'Preguntas con discriminación ≤ 0 (candidatas a eliminación):')
bad_disc

## Conclusions of the question analysis

1. **Very difficult questions** (< 25% correct) are candidates for removal or revision, since their correct-answer rate is below random chance (25% with 4 options).
2. **Questions with negative or zero discrimination** do not serve their evaluative purpose: getting them right does not correlate with knowing more. This may be due to:
   - Confusing or ambiguous wording
   - Every student gets it right (100%) → does not discriminate
   - No student gets it right (0%) → does not discriminate either
   - The correct answer confuses the best students

3. **Questions with good discrimination** (> 0.3) are the most valuable for assessment: they effectively distinguish better-prepared students.
4. The stacked bar chart allows quickly identifying patterns: questions with many errors (red) vs many blanks (grey) indicate different problems (confusion vs ignorance).